# 1、先用20_TSS富集热图.R脚本提取基因+-5kb的信息文件，生成.bed文件

# 2、deeptools 批量计算矩阵

In [2]:
pwd

'/home/weili/Project/AML/human/AML_combined_analyse/0.画图代码'

将下列的代码保存为run_deeptools.sh脚本,在Linux中运行

In [ ]:
#!/bin/bash

BW_DIR="/home/weili/Project/AML/human/AML_combined_analyse/0.画图代码/20_chipseq_特征基因TSS热图/bw/"

# ==================== mRNA/miRNA/lncRNA（TSS 参考点）====================

# Upregulated genes
computeMatrix reference-point \
    -S ${BW_DIR}GSM4604255_576_ATAC-seq.10bp.RPGC.bigwig ${BW_DIR}ATACseq_Control_1.bw \
    -R ./all_up_gene.bed \
    -a 5000 -b 5000 --referencePoint=TSS \
    -o Up_gene_matrix.gz

# Downregulated genes
computeMatrix reference-point \
    -S ${BW_DIR}GSM4604255_576_ATAC-seq.10bp.RPGC.bigwig ${BW_DIR}ATACseq_Control_1.bw \
    -R ./all_down_gene.bed \
    -a 5000 -b 5000 --referencePoint=TSS \
    -o Down_gene_matrix.gz

# 绘图 - 基因
plotHeatmap -m Up_gene_matrix.gz -o Up_gene_heatmap.pdf \
    --colorList 'white,red,#830101' \
    --zMin 0 --zMax 1 \
    --heatmapHeight 15 \
    --heatmapWidth 5 \
    --yAxisLabel "Genes" \
    --regionsLabel "Upregulated" \
    --samplesLabel "H3K27ac" "Input"

plotHeatmap -m Down_gene_matrix.gz -o Down_gene_heatmap.pdf \
    --colorList 'white,red,#830101' \
    --zMin 0 --zMax 1 \
    --heatmapHeight 15 \
    --heatmapWidth 5 \
    --yAxisLabel "Genes" \
    --regionsLabel "Downregulated" \
    --samplesLabel "H3K27ac" "Input"


# ==================== eRNA（增强子中心参考点）====================

# Upregulated eRNA
computeMatrix reference-point \
    -S ${BW_DIR}GSM4604255_576_ATAC-seq.10bp.RPGC.bigwig ${BW_DIR}ATACseq_Control_1.bw \
    -R ./all_up_enh.bed \
    -a 5000 -b 5000 --referencePoint=center \
    -o Up_enh_matrix.gz

# Downregulated eRNA
computeMatrix reference-point \
    -S ${BW_DIR}GSM4604255_576_ATAC-seq.10bp.RPGC.bigwig ${BW_DIR}ATACseq_Control_1.bw \
    -R ./all_down_enh.bed \
    -a 5000 -b 5000 --referencePoint=center \
    -o Down_enh_matrix.gz

# 绘图 - eRNA
plotHeatmap -m Up_enh_matrix.gz -o Up_enh_heatmap.pdf \
    --colorList 'white,red,#830101' \
    --zMin 0 --zMax 1 \
    --heatmapHeight 15 \
    --heatmapWidth 5 \
    --yAxisLabel "eRNAs" \
    --regionsLabel "Upregulated eRNA" \
    --samplesLabel "H3K27ac" "Input"

plotHeatmap -m Down_enh_matrix.gz -o Down_enh_heatmap.pdf \
    --colorList 'white,red,#830101' \
    --zMin 0 --zMax 1 \
    --heatmapHeight 15 \
    --heatmapWidth 5 \
    --yAxisLabel "eRNAs" \
    --regionsLabel "Downregulated eRNA" \
    --samplesLabel "H3K27ac" "Input"

In [3]:
# 直接画图看看效果

In [ ]:
画合在一起的图

# 3 R中进行画图

上面脚本直接画好图了

In [ ]:
#=================接在你原有R代码末尾，绘图开始=================
rm(list = ls(pattern = "^plt"))
library(ggplot2)
library(cowplot)
library(pheatmap)
library(tidyr)
library(dplyr)

# 1. 导入deeptools输出的matrix.gz文件
# 函数：读取computeMatrix输出gz矩阵
read_deeptools_matrix <- function(file){
  con <- gzfile(file,"rt")
  dat <- read.delim(con,skip=1,header=F)
  close(con)
  # 前6列为注释信息：chr start end name strand ...
  anno <- dat[,1:6]
  mat <- dat[,7:ncol(dat)]
  return(list(anno=anno,mat=mat))
}

# 读取全基因矩阵（All_gene_matrix.gz，shell生成的文件）
res <- read_deeptools_matrix("All_gene_matrix.gz")
mat_raw <- res$mat
gene_anno <- res$anno

# 拆分两个样本：前半列=IRF1，后半列=Input
bin_num <- ncol(mat_raw)/2
mat_IRF1 <- mat_raw[,1:bin_num]
mat_Input <- mat_raw[,(bin_num+1):ncol(mat_raw)]

# 横坐标坐标：-5000~+5000，bin=100bp
pos <- seq(-5000+50,5000-50,by=100) # bin中点
df_profile <- data.frame(
  pos = rep(pos,2),
  value = c(colMeans(mat_IRF1),colMeans(mat_Input)),
  sample = rep(c("MV4-11_IRF1","MV4-11_Input"),each=length(pos))
)

# 2. 顶部折线图（profile富集曲线）
p_profile <- ggplot(df_profile,aes(x=pos,y=value,color=sample))+
  geom_line(linewidth=1)+
  scale_color_manual(values=c("#222288","#ff5599"))+
  geom_vline(xintercept=0,lty=2,lwd=0.6)+
  labs(x="",y="",title="genes")+
  theme_bw()+
  theme(legend.position="top",plot.title=element_text(hjust=0.5),
        panel.grid=element_blank())+
  scale_x_continuous(breaks=c(-5000,0,5000),labels=c("-5.0","TSS","5.0Kb"))

# 3. 热图数据合并、归一（共用colorbar，和原图色阶0~0.35一致）
all_mat <- cbind(mat_IRF1,mat_Input)
colnames(all_mat) <- c(paste0("IRF1_",pos),paste0("Input_",pos))
# 固定色值范围和原图一致：0~0.35
color_break <- seq(0,0.35,length.out=100)
col_heat <- colorRampPalette(c("#002277","#88ccff","#fff8dd","#dd2222"))(length(color_break)-1)

# 分栏注释：左右分组
split_col <- factor(c(rep("MV4-11_IRF1",bin_num),rep("MV4-11_Input",bin_num)),
                    levels=c("MV4-11_IRF1","MV4-11_Input"))

# 绘制热图
p_heat <- pheatmap(all_mat,
                   color=col_heat,
                   breaks=color_break,
                   cluster_rows=F,cluster_cols=F,
                   gaps_col=bin_num, #中间竖线分割两个样本
                   annotation_col=data.frame(Sample=split_col,row.names=colnames(all_mat)),
                   show_rownames=F,show_colnames=F,
                   treeheight_col=0,treeheight_row=0,
                   annotation_names_col=F)

# 4. 拼图：上小下大（上下高度比例1:6，匹配原图）
p_all <- plot_grid(
  p_profile, p_heat[[4]],
  ncol=1,rel_heights=c(1,6),align="v"
)

# 保存图片
ggsave("IRF1_TSS_heatmap.pdf",p_all,width=7,height=12,dpi=300)
ggsave("IRF1_TSS_heatmap.png",p_all,width=7,height=12,dpi=300)

cat("绘图完成！图片已输出至当前目录\n")